In [1]:
#!pip install pyspark==3.5.5 tables snakebite-py3

In [2]:
HDFS_HOST = "192.168.2.31"
HDFS_PORT = 9000
HDFS_BASE = f"hdfs://{HDFS_HOST}:{HDFS_PORT}"
import os
import sys
os.environ['PYSPARK_PYTHON'] = "python3"
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
from pyspark.sql import SparkSession

spark_session :SparkSession = SparkSession.builder \
    .master("spark://192.168.2.31:7077") \
    .appName("spark_preprocess_driver") \
    .config("spark.dynamicAllocation.enabled", True) \
    .config("spark.dynamicAllocation.shuffleTracking.enabled", True) \
    .config("spark.shuffle.service.enabled", False) \
    .config("spark.dynamicAllocation.executorIdleTimeout", "30s") \
    .config("spark.executor.memory", "6g") \
    .config("spark.driver.maxResultSize", "3g") \
    .getOrCreate()
    
spark_context = spark_session.sparkContext
spark_context.setLogLevel("ERROR")

#Get this file by:
#1. running "mkdir snakebite", "cd snakebite", "pip install snakebite-py3 -t .", "zip -r ../snakebite.zip .", then putting its path as an argument below
spark_context.addPyFile("./spark_dependencies/snakebite.zip")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/03/17 22:25:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/03/17 22:25:10 WARN StandaloneSchedulerBackend: Dynamic allocation enabled without spark.executor.cores explicitly set, you may get more executors allocated than expected. It's recommended to set spark.executor.cores explicitly. Please check SPARK-30299 for more details.


In [3]:
from snakebite.client import Client
HDFS_HOST = "192.168.2.31"
HDFS_PORT = 9000
HDFS_BASE = f"hdfs://{HDFS_HOST}:{HDFS_PORT}"

client = Client(HDFS_HOST, HDFS_PORT)


/tmp/spark-1b9e9e3f-6d12-45af-8895-c227ff8cc96c/userFiles-5807adf2-2e45-4d8b-80a9-1954636fc625/snakebite.zip/snakebite/client.py:816: SyntaxWarning: "is not" with a literal. Did you mean "!="?
/tmp/spark-1b9e9e3f-6d12-45af-8895-c227ff8cc96c/userFiles-5807adf2-2e45-4d8b-80a9-1954636fc625/snakebite.zip/snakebite/client.py:816: SyntaxWarning: "is not" with a literal. Did you mean "!="?


In [4]:
starting_directory = "/data/MillionSongSubset"

done = False

#NOTE: in case of large amounts of files, this will likely make the driver run out of memory.
#In that case, recursion has to be implemented manually. 
all_files = list(client.ls(["/data/MillionSongSubset"], recurse=True))

In [5]:
h5_files = []
for file in all_files:
    if file["file_type"] != "f":
        continue
    
    if file["path"].split(".")[-1] == "h5":
        h5_files.append(file["path"])
        

In [ ]:
from hdf5_getters import get_desired
import io
import tables
import tempfile



def get_binary_data(file_path):
    HDFS_HOST = "192.168.2.31"
    HDFS_PORT = 9000
    client = Client(HDFS_HOST, HDFS_PORT)
    binary = b''.join(list(client.cat([file_path]))[0])
    return binary

#transformation using pytables
def get_relevant_metadata_of_song_file(binary_data):
    #Change this according to needs. Check for available fields at the bottom of the hdf5_getters file
    RELEVANT_FIELDS = [
    'artist_name',
    'title',
    'duration',
    'year',
    'song_id'
    ]
    
    file_contents = io.BytesIO(binary_data)
    

    with tempfile.NamedTemporaryFile(delete=True) as temp_file:
        temp_file.write(file_contents.getvalue())
        temp_file_path = temp_file.name
        
        file = tables.open_file(temp_file_path)
        song_metadata = get_desired(file, RELEVANT_FIELDS)

    relevant_data = {}
    
    for field in RELEVANT_FIELDS:
        relevant_data[field] = str(song_metadata[field])
    
    return relevant_data

#transformation using h5py
def get_song_name(binary_data):
    import h5py
    file = h5py.File(binary_data)
    song_title = file["metadata"]["songs"][0][18]
    return song_title

In [ ]:
#Note that only a subset of the 10000 files are used
rdd = spark_context.parallelize(h5_files[0:1000])

In [8]:
rdd.top(10)

['/data/MillionSongSubset/A/D/H/TRADHRX12903CD3866.h5',
 '/data/MillionSongSubset/A/D/H/TRADHPE128F429FE9C.h5',
 '/data/MillionSongSubset/A/D/H/TRADHMY128F14AE89E.h5',
 '/data/MillionSongSubset/A/D/H/TRADHMQ128F932EB96.h5',
 '/data/MillionSongSubset/A/D/H/TRADHLI128F428FEFD.h5',
 '/data/MillionSongSubset/A/D/H/TRADHLD128F4241363.h5',
 '/data/MillionSongSubset/A/D/H/TRADHGV128F4233DE0.h5',
 '/data/MillionSongSubset/A/D/H/TRADHGK128F428A000.h5',
 '/data/MillionSongSubset/A/D/H/TRADHBZ128F932D9E9.h5',
 '/data/MillionSongSubset/A/D/G/TRADGXJ128E078FB3A.h5']

In [9]:
binary_rdd = rdd.map(get_binary_data)

#This has to be done since exporting 3rd party libraries to worker nodes involving
#.h5 files like pytables and h5py has been very difficult. 
#Does mean that scaling will be impacted and limited by driver node. 
binary_data = binary_rdd.collect()


In [10]:
relevant_data = [get_relevant_metadata_of_song_file(d) for d in binary_data]

In [14]:
df = spark_session.createDataFrame(relevant_data)

In [15]:
df.show()

+--------------------+---------+--------------------+--------------------+----+
|         artist_name| duration|             song_id|               title|year|
+--------------------+---------+--------------------+--------------------+----+
|           b'Casual'|218.93179|b'SOMZWCG12A8C13C...| b"I Didn't Mean To"|   0|
|     b'The Box Tops'|148.03546|b'SOCIWDW12A8C13D...|        b'Soul Deep'|1969|
| b'Sonora Santanera'|177.47546|b'SOXVLOJ12AB0189...|  b'Amor De Cabaret'|   0|
|         b'Adam Ant'|233.40363|b'SONHOTT12A8C134...|  b'Something Girls'|1982|
|              b'Gob'|209.60608|b'SOFSOCN12A8C143...|   b'Face the Ashes'|2007|
|b'Jeff And Sheri ...| 267.7024|b'SOYMRWW12A6D4FA...|b'The Moon And I ...|   0|
|          b'Rated R'|114.78159|b'SOMJBYD12A6D4F8...|b'Keepin It Real ...|   0|
|b'Tweeterfriendly...|189.57016|b'SOHKNRJ12A6701D...|     b'Drop of Rain'|   0|
| b'Planet P Project'|269.81832|b'SOIAZJW12AB0185...|       b'Pink World'|1984|
|              b'Clp'|266.39628|b'SOUDSG